In [1]:
# Save mortality by year and for each ensemble

In [5]:
import os
import xarray as xr
import numpy as np
from utils.mortality_utils import att_frac
from utils.mortality_utils import mortality
from utils.utils import get_scenario_config

In [2]:
# === Path config ===
BMR_DIR = "/glade/work/awells/air_quality/BMR/"
POP_DIR = "/glade/work/awells/air_quality/SSP_pop/SSP2/"
O3_DIR = "/glade/work/awells/air_quality/CESM/ozone/OSDMA8_BC/"

In [3]:
# === Load data ===
bmr_file = "GBD_BMR_Country_Mask_COPD_1990-2009.nc"
bmr_path = os.path.join(BMR_DIR, bmr_file)
BMR = xr.open_dataarray(bmr_path)  # three quantiles

pop_file = "ssp2_coarse_grid_annual_2000-2100.nc"
pop_path = os.path.join(POP_DIR, pop_file)
population = xr.open_dataarray(pop_path)

In [4]:
# === Calculate beta for RR GBD 2021 ===

# Relative risk for 10ppb increase in OSDMA8, GBD 2021
RR_10ppb = 1.074  # [95% CI 1.014 – 1.137]
# equation is: RR = e^(beta*(x-TMREL)) where RR_10ppb = e^(10beta)
beta = np.log(RR_10ppb)/10

# TMREL from GBD 2021
TMREL = 32.4  # [95% CI 29.1 – 35.7]

In [6]:
# === Scenario and path config ===
# Set to whatever scenario and model you want
# Function returns error if not recognised
model = "CESM2"
scenario = "SSP245_G6"

config = get_scenario_config(model, scenario)
ensemble_members = config["ensemble_members"]
years = config["years"]

SAVE_DIR = f"/glade/work/awells/air_quality/{model}/mortality/ozone/"

# === Main loop ===
for ens_num in ensemble_members:
    print(f"Processing {scenario}, Ensemble {ens_num:02d}")
    # {years.stop - 1} from OSDMA8 calculation
    dates = f"{years.start}-{years.stop - 1}"

    o3_file = f"OSDMA8_BC_CESM2_{scenario}_{ens_num:02d}_{dates}.nc"
    o3_path = os.path.join(O3_DIR, o3_file)
    o3 = xr.open_dataarray(o3_path)

    # Adjust indices to match (with small tolerance)
    # e.g., max 1e-7 km distance
    o3 = o3.reindex_like(BMR, method="nearest", tolerance=1e-9, fill_value=0)
    population = population.reindex_like(BMR, method="nearest", tolerance=1e-9)

    # Flag if any nans present (i.e. reindex was out of tolerance distance)
    assert not population.isnull().any()

    M = []

    for year in o3["year"].values:
        print(f"Processing year {year}")
        POP = population.sel(year=year)
        AF = att_frac(o3.sel(year=year), TMREL, beta)
        mortality_year = mortality(AF, BMR, POP)
        M.append(mortality_year)

        global_mortality = mortality_year.sum(dim=("lat", "lon"))
        mean_mortality = global_mortality.sel(quantile="mean").round().values
        print(f"Mean mortality rate for {year} is {mean_mortality}")

    M_cleaned = [da.drop_vars("year", errors="ignore") for da in M]
    mortality_timeseries = xr.concat(
        M_cleaned,
        dim=(xr.DataArray(o3["year"].values,
                          dims="year", name="year")))

    out_file = f"Mortality_CESM2_{scenario}_{ens_num:02d}_{dates}.nc"
    out_path = os.path.join(SAVE_DIR, out_file)

    print(f"Saving mortality timeseries to {out_path}")
    description = ("Total COPD mortality due to surface ozone - scripts "
                   "by A.F. Wells (2025)")
    mortality_timeseries.attrs["description"] = description
    mortality_timeseries.attrs["ensemble_number"] = ens_num
    mortality_timeseries.attrs["scenario"] = scenario
    mortality_timeseries.to_netcdf(out_path)

Processing SSP245_G6, Ensemble 01
Processing year 2020
Mean mortality rate for 2020 is 446589.0
Processing year 2021
Mean mortality rate for 2021 is 456260.0
Processing year 2022
Mean mortality rate for 2022 is 465712.0
Processing year 2023
Mean mortality rate for 2023 is 456475.0
Processing year 2024
Mean mortality rate for 2024 is 481051.0
Processing year 2025
Mean mortality rate for 2025 is 481587.0
Processing year 2026
Mean mortality rate for 2026 is 480072.0
Processing year 2027
Mean mortality rate for 2027 is 485884.0
Processing year 2028
Mean mortality rate for 2028 is 508658.0
Processing year 2029
Mean mortality rate for 2029 is 481867.0
Processing year 2030
Mean mortality rate for 2030 is 510370.0
Processing year 2031
Mean mortality rate for 2031 is 514936.0
Processing year 2032
Mean mortality rate for 2032 is 497860.0
Processing year 2033
Mean mortality rate for 2033 is 504209.0
Processing year 2034
Mean mortality rate for 2034 is 498030.0
Processing year 2035
Mean mortality 

FileNotFoundError: [Errno 2] No such file or directory: '/glade/work/awells/air_quality/CESM/ozone/OSDMA8_BC/OSDMA8_BC_CESM2_SSP245_G6_04_2020-2083.nc'